<a href="https://colab.research.google.com/github/cxxc7/Gen_AI_Lab/blob/main/GenAI_Prog10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this first. Installs pdf parsing, sentence-transformers and faiss.
!pip install -q pdfplumber sentence-transformers faiss-cpu

In [ ]:
from google.colab import files
import os

# Inform the user to upload the PDF file.
print("Please upload your IPC.pdf file when prompted.")

# Prompt the user to upload the PDF file.
uploaded = files.upload()  # pick IPC.pdf from your computer

# Ensure the uploaded filename is exactly IPC.pdf (or adapt below).
# If a different file name is uploaded, it will be renamed to IPC.pdf.
if "IPC.pdf" in uploaded:
    PDF_PATH = "IPC.pdf"
else:
    # If user uploaded a different name, pick first file and rename to IPC.pdf
    first_name = list(uploaded.keys())[0]
    os.rename(first_name, "IPC.pdf")
    PDF_PATH = "IPC.pdf"

print("Using PDF path:", PDF_PATH)
print("Files in working directory:", os.listdir("."))

Please upload your IPC.pdf file when prompted.


Saving IPC.pdf to IPC (1).pdf
Using PDF path: IPC.pdf
Files in working directory: ['.config', 'IPC.pdf', 'sample_data']


In [ ]:
import pdfplumber
from pathlib import Path

def extract_pages(pdf_path):
    """Extracts text content page by page from a PDF file."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            # Extract text from the current page.
            text = page.extract_text() or ""
            # Perform basic cleaning by stripping leading/trailing whitespace.
            text = text.strip()
            pages.append({"page": i+1, "text": text})
    return pages

# Extract pages from the uploaded PDF.
pages = extract_pages(PDF_PATH)
print(f"Extracted {len(pages)} pages.")

# Print a short preview (first 800 chars) of page 1 for verification.
print("Page 1 preview:\n", pages[0]["text"][:800])

Extracted 119 pages.
Page 1 preview:
 THE INDIAN PENAL CODE
___________
ARRANGEMENT OF SECTIONS
__________
CHAPTER I
INTRODUCTION
PREAMBLE
SECTIONS
1. Title and extent of operation of the Code.
2. Punishment of offences committed within India.
3. Punishment of offences committed beyond, but which by law may be tried within, India.
4. Extension of Code to extra-territorial offences.
5. Certain laws not to be affected by this Act.
CHAPTER II
GENERAL EXPLANATIONS
6. Definitions in the Code to be understood subject to exceptions.
7. Sense of expression once explained.
8. Gender.
9. Number.
10. “Man”. “Woman”.
11. “Person”.
12. “Public”.
13. [Omitted.].
14. “Servant of Government”.
15. [Repealed.].
16. [Repealed.].
17. “Government”.
18. “India”.
19. “Judge”.
20. “Court of Justice”.
21. “Public servant”.
22. “Moveable property”.
23.


In [ ]:
import re
from collections import defaultdict

# Regex to detect typical "Section 1" or "1. Title" style headings (heuristic).
# This helps in splitting the document into meaningful raw chunks.
section_re = re.compile(r"(Section\s+\d+\.?|SECTION\s+\d+\.?|^\s*\d+\.\s+)", re.IGNORECASE | re.MULTILINE)

raw_chunks = []
for p in pages:
    text = p["text"]
    if not text:
        continue

    # Find section-like headings on the page.
    headers = list(section_re.finditer(text))
    if not headers:
        # If no headers are found, the entire page is treated as one chunk.
        raw_chunks.append({"page": p["page"], "header": None, "text": text})
    else:
        # Split the page text by each detected header start.
        starts = [m.start() for m in headers] + [len(text)]
        for i, m in enumerate(headers):
            start = m.start()
            end = starts[i+1]
            chunk_text = text[start:end].strip()
            header_text = m.group(0).strip()
            raw_chunks.append({"page": p["page"], "header": header_text, "text": chunk_text})

print("Initial raw chunks created:", len(raw_chunks))
# Show a sample of the first raw chunk for verification.
print("Sample chunk (first):", raw_chunks[0]["header"], "page", raw_chunks[0]["page"])
print(raw_chunks[0]["text"][:500])

Initial raw chunks created: 1585
Sample chunk (first): 1. page 1
1. Title and extent of operation of the Code.


In [ ]:
def chunk_text(text, max_chars=900, overlap=200):
    """Splits a given text into smaller chunks with optional overlap."""
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = max(0, end - overlap) # Move start back by overlap for the next chunk.
        if end >= len(text):
            break
    return chunks

chunks = []
for i, rc in enumerate(raw_chunks):
    # Further split the raw chunks into smaller, more manageable pieces.
    # `max_chars` and `overlap` are chosen to optimize for embedding quality.
    smalls = chunk_text(rc["text"], max_chars=800, overlap=150)
    for j, s in enumerate(smalls):
        chunks.append({
            "id": f"page{rc['page']}_chunk{i}_{j}", # Unique ID for each final chunk.
            "page": rc["page"],
            "header": rc["header"],
            "text": s
        })

print("Total final chunks:", len(chunks))
print("Example chunk metadata:", chunks[0]["id"], chunks[0]["page"], chunks[0]["header"])
print(chunks[0]["text"][:400])

Total final chunks: 1753
Example chunk metadata: page1_chunk0_0 1 1.
1. Title and extent of operation of the Code.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Using a small, fast, and good quality pre-trained model for embeddings.
model_name = "all-MiniLM-L6-v2"
print("Loading embedding model:", model_name)
embed_model = SentenceTransformer(model_name)

# Prepare the text chunks for embedding.
texts = [c["text"] for c in chunks]
print("Computing embeddings for", len(texts), "chunks...")

# Generate embeddings for all text chunks.
embs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
# Convert embeddings to float32 for FAISS compatibility and memory efficiency.
embs = embs.astype("float32")

print("Embeddings shape:", embs.shape)

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Computing embeddings for 1753 chunks...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

Embeddings shape: (1753, 384)


In [ ]:
import faiss
import json

def normalize_vectors(v):
    """Normalizes a set of vectors to unit length for cosine similarity (Inner Product)."""
    norms = np.linalg.norm(v, axis=1, keepdims=True)
    norms[norms == 0] = 1.0 # Avoid division by zero for zero vectors.
    return v / norms

# Normalize the generated embeddings.
embs_norm = normalize_vectors(embs)

# Determine the dimensionality of the embeddings.
dim = embs_norm.shape[1]

# Create a FAISS index for Inner Product (IP) search.
# Inner product on normalized vectors is equivalent to cosine similarity.
index = faiss.IndexFlatIP(dim)

# Add the normalized embeddings to the FAISS index.
index.add(embs_norm)
print("FAISS index built. Number of vectors:", index.ntotal)

# Save metadata (page, header) associated with each chunk.
# This metadata is crucial for providing citations in the RAG system.
metadata = [{
    "id": c["id"],
    "page": c["page"],
    "header": c["header"]
} for c in chunks]

# Save the metadata to a JSON file.
with open("ipc_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print("Saved ipc_metadata.json")

FAISS index built. Number of vectors: 1753
Saved ipc_metadata.json


In [ ]:
def search_ipc(query, k=5):
    """Searches the FAISS index for the most relevant chunks based on a query."""
    # Encode the query into an embedding.
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb.astype("float32")
    # Normalize the query embedding.
    q_emb = normalize_vectors(q_emb)

    # Perform a similarity search in the FAISS index.
    # D: distances (scores), I: indices of the nearest neighbors.
    D, I = index.search(q_emb, k)
    results = []
    for score, idx in zip(D[0], I[0]):
        if idx < 0:
            continue # Skip invalid indices.
        results.append({
            "score": float(score), # Similarity score.
            "text": texts[idx],     # The retrieved text content.
            "metadata": metadata[idx] # Associated metadata for citation.
        })
    return results

# Quick test search to demonstrate functionality.
print("Test search for 'murder punishment':")
res = search_ipc("murder punishment", k=4)
for r in res:
    print("SCORE", r["score"], "PAGE", r["metadata"]["page"], "HEADER", r["metadata"]["header"])
    # Print a truncated version of the text snippet.
    print(r["text"][:300].replace("\n"," "), "...\n")

Test search for 'murder punishment':
SCORE 0.6768350005149841 PAGE 74 HEADER 303.
303. Punishment for murder by life-convict.—Whoever, being under sentence of 1[imprisonment for life], commits murder shall be punished with death. ...

SCORE 0.6624846458435059 PAGE 8 HEADER 303.
303. Punishment for murder by life-convict. ...

SCORE 0.6575265526771545 PAGE 74 HEADER 302.
302. Punishment for murder.—Whoever commits murder shall be punished with death, or 1[imprisonment for life], and shall also be liable to fine. ...

SCORE 0.654863715171814 PAGE 76 HEADER 307.
307. Attempt to murder.—Whoever does any act with such intention or knowledge, and under such circumstances that, if he by that act caused death, he would be guilty of murder, shall be punished with imprisonment of either description for a term which may extend to ten years, and shall also be liable ...



In [ ]:
def answer_with_citations(question, top_k=5):
    """Retrieves relevant information and formats it as an answer with citations."""
    # Search for top_k most relevant chunks.
    hits = search_ipc(question, k=top_k)

    # Build a compact, readable answer using retrieved text snippets.
    answer_parts = []
    citations = []
    for h in hits:
        md = h["metadata"]
        page = md.get("page")
        header = md.get("header") or "" # Handle cases where header might be None.
        snippet = h["text"].replace("\n", " ").strip()
        # Keep a short snippet for display.
        snippet_short = snippet[:600] + ("..." if len(snippet) > 600 else "")
        answer_parts.append(f"[Page {page}] {header}\n{snippet_short}")
        citations.append(f"Page {page} {header}".strip())

    # Combine all answer parts and sort unique citations.
    answer_text = "\n\n".join(answer_parts)
    citation_text = "\n".join(sorted(set(citations)))
    return {"question": question, "answer": answer_text, "citations": citation_text}

# Demo the answer generation with a sample question.
demo = answer_with_citations("What is the punishment for murder?", top_k=4)
print("QUESTION:", demo["question"])
print("\nANSWER (retrieved snippets):\n", demo["answer"][:1500]) # Print a truncated answer.
print("\nCITATIONS:\n", demo["citations"])

QUESTION: What is the punishment for murder?

ANSWER (retrieved snippets):
 [Page 74] 302.
302. Punishment for murder.—Whoever commits murder shall be punished with death, or 1[imprisonment for life], and shall also be liable to fine.

[Page 74] 303.
303. Punishment for murder by life-convict.—Whoever, being under sentence of 1[imprisonment for life], commits murder shall be punished with death.

[Page 8] 303.
303. Punishment for murder by life-convict.

[Page 76] 307.
307. Attempt to murder.—Whoever does any act with such intention or knowledge, and under such circumstances that, if he by that act caused death, he would be guilty of murder, shall be punished with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine; and if hurt is caused to any person by such act, the offender shall be liable either to 1[imprisonment for life], or to such punishment as is hereinbefore mentioned. Attempts by life-convicts.— 2[When any person offe

In [ ]:
print("IPC Chatbot — type your question. Type 'exit' or 'quit' to stop.")
while True:
    q = input("\nYou: ").strip()
    if q.lower() in ("exit", "quit"):
        print("Exiting.")
        break
    # Get response from the RAG system.
    resp = answer_with_citations(q, top_k=5)
    print("\nAssistant (retrieved passages):\n")
    print(resp["answer"])
    print("\nCITATIONS:\n" + resp["citations"])
    print("\n" + "-"*60)

IPC Chatbot — type your question. Type 'exit' or 'quit' to stop.

You: what is the punishment for murder

Assistant (retrieved passages):

[Page 74] 303.
303. Punishment for murder by life-convict.—Whoever, being under sentence of 1[imprisonment for life], commits murder shall be punished with death.

[Page 74] 302.
302. Punishment for murder.—Whoever commits murder shall be punished with death, or 1[imprisonment for life], and shall also be liable to fine.

[Page 76] 307.
307. Attempt to murder.—Whoever does any act with such intention or knowledge, and under such circumstances that, if he by that act caused death, he would be guilty of murder, shall be punished with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine; and if hurt is caused to any person by such act, the offender shall be liable either to 1[imprisonment for life], or to such punishment as is hereinbefore mentioned. Attempts by life-convicts.— 2[When any person 

## 1. Setup and Installation

This section installs all the necessary Python libraries required for PDF parsing, embedding generation, and vector indexing.

## 2. PDF Upload and Text Extraction

Here, we handle the upload of the `IPC.pdf` file and then use `pdfplumber` to extract all the text content, page by page.

## 3. Text Chunking

This step processes the extracted text. It identifies potential section headers using regular expressions and then further divides the text into smaller, overlapping chunks suitable for embedding.

## 4. Embedding Generation

We use a pre-trained `SentenceTransformer` model (`all-MiniLM-L6-v2`) to convert each text chunk into a numerical vector (embedding). These embeddings capture the semantic meaning of the text.

## 5. FAISS Indexing and Metadata Storage

The generated embeddings are normalized and then indexed using FAISS (Facebook AI Similarity Search) for efficient similarity retrieval. We also store associated metadata (like page number and header) for each chunk.

## 6. RAG Query Functions

These functions define how we search the FAISS index with a user query and how we format the retrieved information into a coherent answer with citations.

## 7. Interactive Chatbot

This section provides a simple interactive loop that allows you to ask questions and receive answers from the indexed IPC document.

## 8. Save FAISS Index

Finally, the FAISS index is saved to disk, so it can be reloaded later without needing to re-process the PDF and re-generate embeddings.

In [ ]:
import faiss

# Save the FAISS index to disk for future use.
# This allows reloading the index without re-computing embeddings.
faiss.write_index(index, "ipc_faiss.index")
print("Saved FAISS index to ipc_faiss.index (and ipc_metadata.json contains metadata).")

Saved FAISS index to ipc_faiss.index (and ipc_metadata.json contains metadata).


## Summary of the IPC Chatbot Approach

This notebook builds a simple Retrieval Augmented Generation (RAG) system to answer questions about the Indian Penal Code (IPC) from a PDF document. The process involves several key steps:

1.  **Dependency Installation**: Install necessary libraries like `pdfplumber`, `sentence-transformers`, and `faiss-cpu`.
2.  **PDF Upload and Extraction**: Upload the `IPC.pdf` file and extract its textual content page by page using `pdfplumber`.
3.  **Text Chunking**: The extracted text is then divided into smaller, manageable chunks. A heuristic approach using regular expressions is employed to identify section headings, and pages are further split to create more granular chunks, ensuring each chunk is within a reasonable character limit with some overlap.
4.  **Embedding Generation**: Each text chunk is converted into a numerical vector (embedding) using the `all-MiniLM-L6-v2` sentence transformer model. These embeddings capture the semantic meaning of the text.
5.  **FAISS Indexing**: The generated embeddings are normalized and then indexed using FAISS (Facebook AI Similarity Search). A `IndexFlatIP` (Inner Product) index is used, which effectively performs cosine similarity search on normalized vectors.
6.  **Metadata Storage**: Along with the embeddings, metadata for each chunk (page number, original header) is stored to allow for proper citation and context in the search results.
7.  **Search Function (`search_ipc`)**: A function is defined to take a query, convert it into an embedding, search the FAISS index for the most similar chunks, and return the relevant text snippets and their metadata.
8.  **Answer Generation (`answer_with_citations`)**: This function orchestrates the search, retrieves the top-k most relevant chunks, and formats them into a readable answer with clear citations (page number and section header).
9.  **Interactive Chatbot**: Finally, an interactive loop is provided to allow users to ask questions and receive answers based on the indexed IPC document.